In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np


classes = ("самолет", "автомобиль", "птица", "кот", "олень", "собака", "лягушка", "лошадь", "корабль", "грузовик")
transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("Количество изображений в обучающей выборке:", len(train_dataset))
print("Количество изображений в тестовой выборке:", len(test_dataset))

image, label = train_dataset[0]
print("Форма одного изображения:", image.shape)
print("Метка изображения:", label)
print("Название класса:", classes[label])

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    image = image.permute(1, 2, 0)
    image = image * 0.5 + 0.5
    ax.imshow(image.numpy())
    ax.set_title(classes[label])
    ax.axis("off")

plt.suptitle("Примеры изображений CIFAR-10")
plt.tight_layout()
plt.show()


batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Количество обучающих пакетов:", len(train_loader))
print("Количество тестовых пакетов:", len(test_loader))

images, labels = next(iter(train_loader))
print("Форма пакета изображений:", images.shape)
print("Форма пакета меток:", labels.shape)


class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)

        return x

model = CNN()
images, labels = next(iter(train_loader))
images = images
outputs = model(images)

print("Форма выходных данных:", outputs.shape)
print(model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

print("Функция потерь:", criterion)
print("Оптимизатор:", optimizer)

model = CNN().to()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

images, labels = next(iter(train_loader))

images = images.to()
outputs = model(images)

print("Форма выходных данных:", outputs.shape)
print(model)


train_losses = []
train_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to()
        labels = labels.to()
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(
        f"Эпоха [{epoch + 1}/{num_epochs}] "
        f"Ошибка: {epoch_loss:.4f} "
        f"Точность: {epoch_accuracy:.2f}%"
    )


model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to()
        labels = labels.to()
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_loss = test_loss / total
test_accuracy = 100 * correct / total

print("\nРЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ")
print("Ошибка классификации:", round(test_loss, 4))
print("Точность классификации:", round(test_accuracy, 2), "%")


plt.figure(figsize=(10, 5))

plt.plot(
    range(1, num_epochs + 1),
    train_losses,
    marker="o",
    label="Ошибка на обучающей выборке"
)

plt.xlabel("Эпоха")
plt.ylabel("Ошибка (Loss)")
plt.title("Изменение ошибки во время обучения")

plt.xticks(range(1, num_epochs + 1))
plt.grid(True)
plt.legend()

plt.show()
